# Compare automated extraction with manual validation

Compare **automated extraction** from `Open_Alex_Match.py` with **manual
validation** in
`data/reference/author_relations_gold_standard_manual_validation.csv`.
These names are used throughout for the two datasets. Automated extraction
is read from the path configured by `output_filename_gold_standard`.

This notebook follows the setup, inspection, and plotting pattern of
`clean_data.ipynb`. It runs offline using the saved automated extraction
and manual validation; run `Open_Alex_Match.py` separately to evaluate
a new automated extraction.

Run all cells from top to bottom. This notebook exports its diagnostic
tables and figures to `data/output/gold_standard_diagnostics/` when
`WRITE_REPORTS` is enabled. It does not edit manual validation or automated
extraction.

## 1. Setup and comparison rules

Align manual validation and automated extraction by thesis `integer_id`.
First compare PhD IDs, then supervisor IDs, and finally complete PhD-supervisor
links. Compare supervisor IDs as sets within each thesis, across all five
supervisor slots in manual validation. Contributor rank in automated
extraction is not assumed to equal supervisor order in manual validation.

Use the confirmation decisions in automated extraction: nonempty
`phd_match_by` for PhDs and true `contributor_confirmed` for contributors.
In this notebook, an accepted ID is an ID confirmed by automated extraction;
it is correct only if it also agrees with manual validation.

Count each thesis once and each distinct supervisor ID once per thesis.
Evaluate every ID when automated extraction returns several different IDs
for one contributor. Missing IDs in manual validation remain unknown and
are excluded from the relevant measures. Fractions with zero denominators
are shown as undefined.

Use theses present in both manual validation and automated extraction for
the matching measures. Disagreements can
reflect errors in manual validation or automated extraction, or changes
to OpenAlex profiles between their respective dates.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

# Also works when Jupyter starts in a subdirectory of the repository.
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "pyproject.toml").exists() and (p / "src/plotters.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside this repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.gold_standard_diagnostics import (
    PHD_OUTCOMES, clean_manual_identifiers, compare_identities, coverage_table,
    export_frame, group_metrics, metric_summary, prepare_automated,
    prepare_manual, ratio, recover_manual_keys,
)
from src.plotters import ContributorMatchPlotter, PhDMatchPlotter

pd.set_option("display.max_columns", 18)
pd.set_option("display.max_colwidth", 85)
pd.set_option("display.max_rows", 15)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
CONFIG_PATH = ROOT / "dataset_config.yaml"
config = (yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8")) or {}) if CONFIG_PATH.exists() else {}

# Select automated extraction for the manual validation sample, even if the config selects a full run.
# Set AUTOMATED_EXTRACTION_PATH below to compare a particular automated extraction.
AUTOMATED_EXTRACTION_PATH = ROOT / config.get(
    "output_filename_gold_standard", "data/output/author_relations_gold_standard.csv"
)
MANUAL_VALIDATION_PATH = ROOT / "data/reference/author_relations_gold_standard_manual_validation.csv"
AUTOMATED_EXTRACTION_INPUT_PATH = ROOT / config.get(
    "dataset_path_sample_gold_standard", "data/output/sampled_pubs_cleaned.csv"
)
REPORT_DIR = ROOT / "data/output/gold_standard_diagnostics"
WRITE_REPORTS = True
PREVIEW_ROWS = 12

# Manual validation notes for these theses explicitly say that supervisor 2 was misidentified.
# Their alternative IDs REPLACE supervisor 2's recorded ID, rather than the PhD's ID.
# Review this mapping when replacing manual validation. Unmapped alternatives are audited.
ALTERNATIVE_SUPERVISOR_SLOTS = {"110543": 2, "64857": 2}

for path in (AUTOMATED_EXTRACTION_PATH, MANUAL_VALIDATION_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Required input does not exist: {path}")

tables, figures = {}, {}

# Presentation labels identify manual validation and automated extraction;
# calculations retain the column names expected by the comparison helpers.
SHARED_THESES = "Theses in both manual validation and automated extraction"
METRIC_LABELS = {
    "n_theses": "Theses in comparison",
    "n_extracted": "Theses present in automated extraction",
    "n_phd_labeled": "Theses with PhD IDs in manual validation",
    "n_phd_confirmed": "PhDs confirmed by automated extraction with manual validation IDs available",
    "n_phd_correct": "Confirmed automated extraction PhD IDs matching manual validation",
    "n_phd_wrong": "Confirmed automated extraction PhD IDs differing from manual validation",
    "phd_precision": "PhD precision",
    "phd_recall": "PhD recall",
    "phd_f1": "PhD F1",
    "phd_candidate_recall": "Candidate recall",
    "n_relation_theses": "Theses eligible for supervisor/link comparison",
    "n_reference_relations": "Supervisor pairs in manual validation",
    "n_predicted_relations": "Confirmed supervisor pairs in automated extraction",
    "n_identity_hits": "Automated extraction supervisor pairs matching manual validation",
    "n_link_hits": "Automated extraction PhD-supervisor links matching manual validation",
    "supervisor_id_precision": "Individual supervisor precision",
    "supervisor_id_recall": "Individual supervisor recall",
    "supervisor_id_f1": "Individual supervisor F1",
    "all_supervisors_precision": "All supervisors precision",
    "all_supervisors_recall": "All supervisors recall",
    "link_precision": "End-to-end link precision",
    "link_recall": "End-to-end link recall",
    "link_f1": "End-to-end link F1",
    "exact_supervisor_set_rate": "Automated extraction supervisor set equals manual validation",
    "all_supervisors_found_rate": "Automated extraction includes every manual validation supervisor",
}
OUTCOME_LABELS = {
    "Correct confirmed ID": "Automated extraction ID matches manual validation",
    "Wrong confirmed ID": "Automated extraction ID differs from manual validation",
    "Not confirmed": "Not confirmed by automated extraction",
    "Missing extraction": "Thesis absent from automated extraction",
    "No manual PhD ID (unscored)": "PhD ID missing in manual validation (unscored)",
    "Reference IDs only": "All IDs match manual validation",
    "Reference hit + extra IDs": "Matches manual validation + additional IDs",
    "No reference hit": "No ID matches manual validation",
    "Confirmed without ID": "Confirmed without ID in automated extraction",
    "Incomplete reference (unscored)": "Manual validation incomplete (unscored)",
    "Reference hits": "Automated extraction matches manual validation",
    "Extra / incorrect predictions": "Automated extraction disagrees with manual validation",
    "Missed reference relations": "Manual validation relations missed by automated extraction",
    "Exact supervisor set": "Automated extraction set equals manual validation",
    "All reference IDs + extra IDs": "All manual validation IDs + additional automated extraction IDs",
    "Some reference supervisors found": "Some manual validation supervisors found by automated extraction",
    "No reference hits": "No automated extraction IDs match manual validation",
    "No accepted supervisor IDs": "No confirmed supervisor IDs in automated extraction",
    "both": "In manual validation and automated extraction",
    "left_only": "In manual validation only",
    "right_only": "In automated extraction only",
}
FIELD_LABELS = {
    "in_automated": "Thesis present in automated extraction",
    "phd_correct": "Automated extraction PhD ID agrees with manual validation",
    "phd_outcome": "Automated extraction PhD ID compared with manual validation",
    "identity_outcome": "Automated extraction contributor IDs compared with manual validation",
    "phd_candidate_hit": "Automated extraction includes a PhD ID from manual validation",
    "phd_orcid_comparable": "Usable PhD ORCIDs in manual validation and automated extraction",
    "phd_orcid_agrees": "PhD ORCIDs agree between manual validation and automated extraction",
    "correct_supervisor_ids": "Automated extraction supervisor IDs matching manual validation",
    "missed_supervisor_ids": "Manual validation supervisor IDs missed by automated extraction",
    "extra_supervisor_ids": "Automated extraction supervisor IDs absent from manual validation",
    "correct_link_ids": "Automated extraction links matching manual validation",
    "missed_link_ids": "Manual validation links missed by automated extraction",
    "exact_supervisor_set": "Automated extraction supervisor set equals manual validation",
    "all_reference_supervisors_found": "Automated extraction includes every manual validation supervisor",
    "matched_reference_ids": "Automated extraction IDs matching manual validation",
    "extra_candidate_ids": "Automated extraction candidate IDs absent from manual validation",
    "n_candidate_ids": "Candidate IDs in automated extraction",
    "n_manual_supervisors": "Supervisors in manual validation",
    "n_unresolved_supervisors": "Named supervisors without IDs in manual validation",
    "n_reference_ids": "Supervisor IDs in manual validation",
    "n_predicted_ids": "Confirmed supervisor IDs in automated extraction",
    "evidence_category": "Evidence category in automated extraction",
    "phd_evidence_category": "PhD evidence category in automated extraction",
    "coverage": "Presence in manual validation and automated extraction",
    "n_contributor_rows": "Contributor rows in automated extraction",
    "n_confirmed_contributor_rows": "Confirmed contributor rows in automated extraction",
    "n_confirmed": "Confirmed contributor rows in automated extraction",
    "n_confirmed_with_complete_reference": "Confirmed automated extraction rows with resolved manual validation lists",
    "n_reference_hit_rows": "Automated extraction rows with a manual validation ID match",
    "reference_hit_rate_among_scored_confirmations": "Contributor match rate against manual validation",
    "n_confirmed_multi_ID_rows": "Confirmed multi-ID rows in automated extraction",
    "in_cleaned_sample": "Thesis present in input used by automated extraction",
    "cleaned_phd_name_missing": "PhD name missing in input used by automated extraction",
    "deduplicated_into_integer_id": "Retained thesis ID after duplicate removal in automated extraction",
    "accessibility": "Thesis accessibility in manual validation",
    "manual_notes": "Notes in manual validation",
    "original": "Original value in manual validation",
    "normalized": "Normalized value in manual validation",
    "original_id": "Supervisor ID in manual validation before correction",
    "alternative_id": "Supervisor ID correction from manual validation",
    "minimum_score": "Minimum automated extraction PhD score",
    "n_retained": "Confirmed automated extraction PhDs retained",
    "n_correct": "Retained automated extraction PhD IDs matching manual validation",
    "n_wrong": "Retained automated extraction PhD IDs differing from manual validation",
    "precision": "Automated extraction PhD precision against manual validation",
    "recall": "Automated extraction PhD recall against manual validation",
}

def dataset_label(value, origin="manual validation"):
    if not isinstance(value, str):
        return value
    if value in OUTCOME_LABELS:
        return OUTCOME_LABELS[value]
    if value in METRIC_LABELS:
        return METRIC_LABELS[value]
    if value in FIELD_LABELS:
        return FIELD_LABELS[value]
    if value.startswith("reference_"):
        return "manual validation " + value.removeprefix("reference_").replace("_", " ")
    for prefix in ("auto_", "candidate_", "predicted_"):
        if value.startswith(prefix):
            return "automated extraction " + value.removeprefix("auto_").replace("_", " ")
    if value == "integer_id":
        return "Thesis integer_id"
    if value in automated_extraction.columns and (origin == "automated extraction" or value not in manual_validation.columns):
        return "automated extraction " + value.replace("_", " ")
    if value in manual_validation.columns:
        return "manual validation " + value.replace("_", " ")
    return value

def dataset_table(frame, origin="manual validation"):
    result = export_frame(frame)
    for column in ("phd_outcome", "identity_outcome", "coverage", "supervisor_set_outcome"):
        if column in result:
            result[column] = result[column].map(lambda value: OUTCOME_LABELS.get(value, value))
    result = result.rename(columns=lambda value: dataset_label(value, origin),
                           index=lambda value: dataset_label(value, origin))
    return result.rename_axis(index=dataset_label(result.index.name, origin),
                              columns=dataset_label(result.columns.name, origin))

TABLE_TITLES = {
    "input_sizes": "Manual validation and automated extraction sizes",
    "confirmation_coverage_counts": "Confirmation coverage counts in manual validation and automated extraction",
    "recovered_manual_keys": "Manual validation thesis IDs recovered from automated extraction",
    "identifier_quality_audit": "Manual validation identifier repairs and exclusions",
    "alternative_supervisor_id_audit": "Manual validation supervisor ID corrections",
    "input_provenance": "Manual validation and automated extraction files used",
    "cohort_coverage": "Thesis coverage in manual validation and automated extraction",
    "cohort_differences": "Theses absent from manual validation or automated extraction",
    "manual_label_completeness": "Manual validation completeness",
    "incomplete_manual_labels": "Manual validation IDs unavailable for comparison",
    "extraction_integrity": "Automated extraction consistency checks",
    "reference_correction_sensitivity": "Effect of manual validation corrections on automated extraction supervisor agreement",
    "link_correction_sensitivity": "Effect of manual validation corrections on automated extraction full-link agreement",
    "missed_manual_validation_links": "Manual validation full links missed by automated extraction",
    "phd_outcomes": "Automated extraction PhD IDs compared with manual validation",
    "phd_outcomes_by_evidence_counts": "Manual validation agreement by automated extraction evidence (counts)",
    "phd_outcomes_by_evidence_fractions": "Manual validation agreement by automated extraction evidence (fractions)",
    "phd_score_retention": "Automated extraction PhD score cutoffs compared with manual validation",
    "supervision_relation_counts": "Automated extraction supervisor and link agreement with manual validation",
    "supervisor_set_outcomes": "Automated extraction supervisor sets compared with manual validation",
    "contributor_identity_by_evidence": "Automated extraction contributor evidence compared with manual validation",
    "contributor_rank_diagnostics": "Automated extraction contributor ranks compared with manual validation",
    "manual_supervisor_count_vs_contributor_scope": "Manual validation supervisor counts versus automated extraction contributor counts",
    "metrics_by_institution": "Automated extraction agreement grouped by manual validation institution",
    "metrics_by_year": "Automated extraction agreement grouped by manual validation year",
    "metrics_by_thesis_accessibility": "Automated extraction agreement grouped by manual validation accessibility",
    "metrics_by_manual_notes": "Automated extraction agreement grouped by manual validation notes",
    "conditional_on_correct_phd": "Automated extraction supervisor agreement after PhD ID agreement with manual validation",
    "phd_orcid_concordance": "PhD ORCID agreement between automated extraction and manual validation",
    "different_openalex_id_same_orcid": "Automated extraction and manual validation with differing OpenAlex IDs but agreeing ORCIDs",
    "phd_disagreements": "Automated extraction PhD disagreements with manual validation",
    "missed_manual_supervisors": "Manual validation supervisors missed by automated extraction",
    "extra_supervisor_predictions": "Automated extraction supervisor IDs absent from manual validation",
    "ambiguous_contributor_identities": "Automated extraction contributors with several accepted IDs",
    "confirmed_contributors_without_reference_hit": "Confirmed automated extraction contributors without a manual validation ID match",
}

def show_table(name, frame, rows=PREVIEW_ROWS, origin="manual validation"):
    tables[name] = frame.copy()
    print(f"{TABLE_TITLES[name]}: {len(frame):,} rows (showing up to {rows})")
    display(dataset_table(frame, origin).head(rows))

PLOT_GUIDANCE = {
    "confirmation_coverage_overview": "Higher confirmation coverage is better for completeness within the list recorded by each dataset; this is separate from correctness against manual validation.",
    "phd_extraction_evidence": "Evidence-category and confirmation counts are descriptive; there is no higher/lower-is-better direction for these counts alone.",
    "recall_overview": "Recall measures recovery of manual validation PhDs, individual supervisors, complete supervisor sets, and full links.",
    "precision_overview": "Precision measures agreement with manual validation among automated extraction PhDs, individual supervisors, supervisor sets, and full links.",
    "phd_identity_outcomes": "Higher PhD-ID agreement is better; lower disagreement and unconfirmed counts are better. Unscored counts describe manual validation completeness.",
    "phd_agreement_by_evidence": "PhD level: higher matching-ID counts are better; lower differing-ID and unconfirmed counts are better. Unscored counts are descriptive.",
    "phd_score_distributions": "Histogram heights are descriptive frequencies, with no higher/lower-is-better direction. Higher scores mean stronger automated extraction evidence, not verified correctness.",
    "phd_score_retention_curve": "Higher precision and higher recall are better. Increasing the score cutoff itself has no single better direction.",
    "contributor_extraction_evidence": "Evidence-category and confirmation counts are descriptive; there is no higher/lower-is-better direction for these counts alone.",
    "contributor_reference_agreement_by_evidence": "More IDs matching manual validation are better; fewer additional or differing confirmed IDs are better. Unconfirmed and unscored contributor counts are descriptive.",
    "supervisor_identity_and_set_agreement": "Higher supervisor-ID agreement and exact-set counts are better; lower incorrect and missed counts are better. Other set categories should decrease as exact sets increase.",
    "end_to_end_link_agreement": "Higher complete-link agreement is better; lower incorrect-link and missed-link counts are better.",
    "recall_by_institution_and_year": "Higher recall is better for PhDs first, supervisor IDs second, and full PhD-supervisor links third.",
}
PLOT_HEADINGS = {
    "confirmation_coverage_overview": "Confirmation coverage in manual validation and automated extraction",
    "phd_extraction_evidence": "PhD-level automated extraction evidence",
    "recall_overview": "Automated extraction recall compared with manual validation",
    "precision_overview": "Automated extraction precision compared with manual validation",
    "phd_identity_outcomes": "PhD-level agreement: automated extraction compared with manual validation",
    "phd_agreement_by_evidence": "PhD-level agreement: automated extraction compared with manual validation",
    "phd_score_distributions": "PhD-level scores: automated extraction compared with manual validation",
    "phd_score_retention_curve": "PhD-level precision and recall: automated extraction compared with manual validation",
    "contributor_extraction_evidence": "Supervisor-level automated extraction evidence",
    "contributor_reference_agreement_by_evidence": "Supervisor-level agreement: automated extraction compared with manual validation",
    "supervisor_identity_and_set_agreement": "Supervisor-level agreement: automated extraction compared with manual validation",
    "end_to_end_link_agreement": "Full-link agreement: automated extraction compared with manual validation",
    "recall_by_institution_and_year": "Automated extraction recall against manual validation: PhDs, supervisors, then full links",
}
HIGHER_IS_BETTER = {
    "PhD precision", "PhD recall", "PhD F1", "Candidate recall",
    "Individual supervisor precision", "Individual supervisor recall", "Individual supervisor F1",
    "All supervisors precision", "All supervisors recall",
    "End-to-end link precision", "End-to-end link recall", "End-to-end link F1",
    "Correct confirmed ID", "Reference IDs only", "Reference hits", "Exact supervisor set",
}
LOWER_IS_BETTER = {
    "Wrong confirmed ID", "Confirmed without ID", "No reference hit", "No reference hits",
    "Extra / incorrect predictions", "Missed reference relations",
}

def sentence_case(value):
    import re
    if not isinstance(value, str) or not value:
        return value
    for pattern, spelling in ((r"\bphd\b", "PhD"), (r"\borcid\b", "ORCID"), (r"\bopenalex\b", "OpenAlex")):
        value = re.sub(pattern, spelling, value, flags=re.IGNORECASE)
    value = re.sub(r"\bOpen\s+Alex\b", "OpenAlex", value, flags=re.IGNORECASE)
    return value[0].upper() + value[1:]

def chart_label(value, name, *, legend=False, include_direction=True):
    if not isinstance(value, str):
        return value
    suffix = ""
    if value.endswith(")") and " (n=" in value:
        value, count = value.rsplit(" (n=", 1)
        suffix = " (n=" + count
    result = sentence_case(dataset_label(value))
    if include_direction and name not in {"phd_extraction_evidence", "contributor_extraction_evidence", "phd_score_distributions"}:
        if value in HIGHER_IS_BETTER or result in HIGHER_IS_BETTER:
            result += " (higher is better)"
        elif value in LOWER_IS_BETTER:
            result += " (lower is better)"
        elif value == "Not confirmed" and name in {"phd_identity_outcomes", "phd_agreement_by_evidence"}:
            result += " (lower is better for recall)"
    if legend and value in {"Confirmed", "Not confirmed"} and name in {"phd_extraction_evidence", "contributor_extraction_evidence"}:
        result = sentence_case(value + " in automated extraction")
    return result + suffix

def show_plot(name, figure):
    from textwrap import fill
    figure.suptitle(PLOT_HEADINGS[name], fontsize=12)
    for ax in figure.axes:
        ax.set_title("")
        ax.set_xlabel(sentence_case(ax.get_xlabel()))
        ax.set_ylabel(sentence_case(ax.get_ylabel()))
        direction_in_axis = "higher is better" in (ax.get_xlabel() + " " + ax.get_ylabel()).lower()
        for axis in (ax.xaxis, ax.yaxis):
            previous = [tick.get_text() for tick in axis.get_majorticklabels()]
            labels = [fill(chart_label(value, name, include_direction=not direction_in_axis), width=43) for value in previous]
            if labels != previous:
                axis.set_ticks(axis.get_majorticklocs(), labels=labels)
                axis.set_tick_params(labelsize=8)
        legend = ax.get_legend()
        if legend:
            for label in legend.get_texts():
                label.set_text(fill(chart_label(label.get_text(), name, legend=True, include_direction=not direction_in_axis), width=44))
                label.set_fontsize(7)
    figure.text(0.5, 0.02, fill(PLOT_GUIDANCE[name], width=140), ha="center", va="bottom", fontsize=8)
    if name == "confirmation_coverage_overview":
        for ax in figure.axes:
            handles, labels = ax.get_legend_handles_labels()
            if ax.get_legend():
                ax.get_legend().remove()
            legend = figure.legend(handles, labels, loc="lower center",
                                   bbox_to_anchor=(0.5, 0.08),
                                   fontsize=8, frameon=False, ncol=1)
            for label in legend.get_texts():
                label.set_text(fill(label.get_text(), width=55))
        figure.tight_layout(rect=(0, 0.22, 1, 0.94))
    else:
        figure.tight_layout(rect=(0, 0.13, 1, 0.94))
    figures[name] = figure
    plt.show()
    plt.close(figure)

## 2. Load automated extraction and audit manual validation

Remove padding rows from manual validation. Recover a missing `integer_id`
in manual validation only through a unique exact `thesis_identifier` match
with automated extraction, and record that recovery. Names are not used
to recover keys.

Normalize OpenAlex URLs and bare IDs in manual validation and automated
extraction for comparison. In manual validation, repair obvious `ttps://`
and copied `orcid":"` URL artifacts and record the changes. Exclude
malformed ORCIDs in manual validation from the ORCID comparison, retaining
their original values in the audit; truncated digits are not inferred.
Unrecognized author IDs, conflicting values within an automated extraction
thesis, and duplicate automated extraction thesis/rank rows raise an error.

In [ ]:
manual_validation_raw = pd.read_csv(MANUAL_VALIDATION_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig")
automated_extraction_raw = pd.read_csv(AUTOMATED_EXTRACTION_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig")
manual_validation_keyed, key_recovery = recover_manual_keys(manual_validation_raw, automated_extraction_raw)
manual_validation_clean, identifier_audit = clean_manual_identifiers(manual_validation_keyed)
manual_validation, alternative_audit = prepare_manual(manual_validation_clean, ALTERNATIVE_SUPERVISOR_SLOTS)
automated_extraction = prepare_automated(automated_extraction_raw)
thesis_comparison, contributor_comparison = compare_identities(manual_validation, automated_extraction)
shared_thesis_comparison = thesis_comparison.loc[thesis_comparison.in_automated].copy()
if shared_thesis_comparison.empty:
    raise ValueError("Manual validation and automated extraction do not share any thesis integer_id values.")

show_table("input_sizes", pd.DataFrame({
    "input": ["manual validation rows including padding", "manual validation theses excluding padding", "automated extraction contributor rows", "automated extraction theses"],
    "rows": [len(manual_validation_raw), len(manual_validation), len(automated_extraction), automated_extraction.integer_id.nunique()],
}))
show_table("recovered_manual_keys", key_recovery)
show_table("identifier_quality_audit", identifier_audit, rows=30)
show_table("alternative_supervisor_id_audit", alternative_audit)

provenance = pd.DataFrame([
    {"input": label, "path": str(path.relative_to(ROOT)) if path.is_relative_to(ROOT) else str(path),
     "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
     "modified_utc": datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).isoformat()}
    for label, path in [("automated extraction", AUTOMATED_EXTRACTION_PATH), ("manual validation", MANUAL_VALIDATION_PATH)]
])
show_table("input_provenance", provenance)

## 3. Check manual validation completeness

A **cohort** is the group of theses included in a comparison. **Cohort
coverage** checks thesis-ID membership in manual validation and automated
extraction. The coverage audit is retained in the reports exported by this
notebook; section 5 gives the difference as a single text output.

**Manual validation completeness** checks whether manual validation has
the IDs needed to score automated extraction. A **resolved supervisor
list** in manual validation is nonempty and has an ID for every named
supervisor; this does not establish that every real supervisor was listed.

In [ ]:
coverage = coverage_table(manual_validation, automated_extraction)
if AUTOMATED_EXTRACTION_INPUT_PATH.is_file():
    automated_extraction_input = pd.read_csv(AUTOMATED_EXTRACTION_INPUT_PATH, dtype=str, keep_default_na=False)
    automated_extraction_input["integer_id"] = automated_extraction_input.integer_id.str.replace(r"\.0+$", "", regex=True).str.strip()
    if not automated_extraction_input.integer_id.duplicated().any() and "phd_name" in automated_extraction_input:
        automated_extraction_input_names = automated_extraction_input.set_index("integer_id").phd_name.str.strip()
        coverage["in_cleaned_sample"] = coverage.integer_id.isin(automated_extraction_input_names.index)
        coverage["cleaned_phd_name_missing"] = coverage.integer_id.map(automated_extraction_input_names).eq("")
coverage_counts = coverage.coverage.value_counts().reindex(["both", "left_only", "right_only"], fill_value=0)
coverage_counts.index = ["In manual validation and automated extraction", "In manual validation only", "In automated extraction only"]
tables["cohort_coverage"] = coverage_counts.rename("n_theses").to_frame()
tables["cohort_differences"] = coverage.loc[coverage.coverage.ne("both")].copy()
tables["full_cohort_coverage"] = coverage

label_quality = pd.DataFrame({
    "annotation": ["PhD ID missing in manual validation", "No supervisor IDs in manual validation", "Named supervisors without IDs in manual validation",
                   "Resolved nonempty supervisor list in manual validation", "More than two named supervisors in manual validation", "Notes present in manual validation"],
    "n_theses": [manual_validation.reference_phd_ids.map(len).eq(0).sum(),
                 manual_validation.reference_supervisor_ids.map(len).eq(0).sum(),
                 manual_validation.n_unresolved_supervisors.gt(0).sum(),
                 manual_validation.reference_supervisors_complete.sum(),
                 manual_validation.n_manual_supervisors.gt(2).sum(), manual_validation.notes.str.strip().ne("").sum()],
})
show_table("manual_label_completeness", label_quality)
show_table("incomplete_manual_labels", thesis_comparison.loc[
    ~thesis_comparison.reference_phd_ids.map(bool) | ~thesis_comparison.reference_supervisors_complete,
    ["integer_id", "author_name", "reference_phd_ids", "reference_supervisor_ids", "n_unresolved_supervisors", "notes"],
])

automated_extraction_phd_rows = automated_extraction.drop_duplicates("integer_id")
integrity = pd.DataFrame({
    "check": ["PhD confirmed without ID in automated extraction", "Contributor confirmed without ID in automated extraction",
              "Contributor confirmed under unconfirmed PhD in automated extraction", "Theses with automated extraction errors"],
    "n": [(automated_extraction_phd_rows.phd_confirmed & automated_extraction_phd_rows.candidate_phd_ids.map(len).eq(0)).sum(),
          (automated_extraction.contributor_confirmed & automated_extraction.candidate_supervisor_ids.map(len).eq(0)).sum(),
          (automated_extraction.contributor_confirmed & ~automated_extraction.phd_confirmed).sum(),
          thesis_comparison.processing_error.ne("").sum()],
})
show_table("extraction_integrity", integrity)

## 4. Automated extraction PhD decisions and evidence

This first plot describes PhD confirmation decisions and evidence from
automated extraction for theses also present in manual validation. Agreement
with manual validation is assessed next. Each thesis contributes one PhD
record. Supervisor results follow the detailed PhD results in section 7;
complete-link results follow supervisor results in section 8.

In [ ]:
automated_extraction_shared = automated_extraction.loc[automated_extraction.integer_id.isin(shared_thesis_comparison.integer_id)].copy()
phd_plot_input = automated_extraction_shared.drop_duplicates("integer_id").copy()
phd_plot_input["phd_name"] = phd_plot_input.integer_id
phd_plotter = PhDMatchPlotter(phd_plot_input)
ax = phd_plotter.plot()
ax.set_title(f"Automated extraction confirms {phd_plotter.n_confirmed} of {phd_plotter.n_phds} PhD records")
show_plot("phd_extraction_evidence", ax.figure)

contributor_plotter = ContributorMatchPlotter(contributor_comparison)

category_map = phd_plotter.df_unique.set_index("phd_name").match_category
thesis_comparison["phd_evidence_category"] = thesis_comparison.integer_id.map(category_map).fillna("Missing extraction")
shared_thesis_comparison = thesis_comparison.loc[thesis_comparison.in_automated].copy()
contributor_comparison["evidence_category"] = contributor_plotter.df.match_category

## 5. Automated extraction agreement with manual validation

**Recall** measures what fraction of IDs in manual validation automated
extraction correctly accepts. **Precision** measures what fraction of IDs
accepted by automated extraction agree with manual validation. **F1** combines
them as `2 × matching IDs / (accepted automated extraction IDs + manual
validation IDs)`.

At the **PhD level**, agreement means the PhD ID confirmed by automated
extraction matches manual validation. At the **supervisor level**, compare
supervisor IDs independently of PhD ID agreement.

A **manual validation match** is an accepted supervisor ID from automated
extraction that occurs in manual validation for the same thesis.

**Individual supervisor** measures count distinct `(integer_id, supervisor
ID)` pairs. **All supervisors** measures count each thesis once and require
the entire set of supervisor IDs confirmed by automated extraction to
equal the supervisor set in manual validation, with no missing or additional
IDs. A partially recovered supervisor list does not count as a correct
complete set. All-supervisors recall uses every eligible thesis;
all-supervisors precision uses only eligible theses with at least one
supervisor ID confirmed by automated extraction. Both compare supervisor
IDs independently of PhD ID agreement.

**End-to-end** refers to the complete matching chain: identifying the PhD,
then identifying that PhD's supervisor. A correct **end-to-end link** requires
both of the following for the same thesis:

1. Automated extraction confirms the PhD ID, and that ID matches manual validation.
2. Automated extraction confirms a supervisor ID for that PhD, and that supervisor ID matches manual validation.

A supervisor ID matching manual validation while the PhD ID differs counts
as a supervisor-ID match, but not as a correct end-to-end link. Count distinct
`(integer_id, supervisor ID)` pairs.

For example, suppose manual validation identifies PhD `A1` and supervisor
`A2` for a thesis. If automated extraction confirms PhD `A1` with supervisor
`A2`, the end-to-end link is correct. If automated extraction instead
confirms PhD `A3` with supervisor `A2`, the supervisor ID agrees with manual
validation, but the end-to-end link is incorrect because the PhD ID differs.

| Measure | Meaning | Calculation |
| --- | --- | --- |
| PhD precision | How reliable the confirmed PhD IDs in automated extraction are against manual validation. | PhD IDs accepted by automated extraction that agree with manual validation / PhDs accepted by automated extraction where manual validation supplies an ID. |
| PhD recall | How completely automated extraction recovers PhD identities in manual validation. | PhD IDs accepted by automated extraction that agree with manual validation / theses with a PhD ID in manual validation. |
| PhD F1 | Balance between automated extraction PhD precision and recall against manual validation. | Twice the matching accepted PhD count / (accepted PhD count in automated extraction + PhD count in manual validation), restricted to theses with a PhD ID in manual validation. |
| Candidate recall | How often automated extraction contains the PhD ID in manual validation, even without confirmation. | Fraction of theses with a PhD ID in manual validation where that ID is present in automated extraction, whether or not automated extraction confirms it. Profiles considered during matching but absent from automated extraction are not counted. |
| Individual supervisor precision | How reliable the accepted supervisor IDs in automated extraction are against manual validation. | Accepted supervisor pairs from automated extraction that match manual validation / accepted supervisor pairs from automated extraction, regardless of PhD ID agreement. |
| Individual supervisor recall | How completely automated extraction recovers supervisor IDs in manual validation. | Accepted supervisor pairs from automated extraction that match manual validation / supervisor pairs in manual validation. |
| Individual supervisor F1 | Balance between automated extraction supervisor-ID precision and recall against manual validation. | Twice the matching supervisor-pair count / (accepted supervisor pairs in automated extraction + supervisor pairs in manual validation). |
| All supervisors recall | How often automated extraction recovers the complete supervisor set in manual validation exactly. | Theses where automated extraction accepts all and only the supervisor IDs in manual validation / all eligible theses. This is also the exact supervisor set rate. |
| All supervisors precision | How often a nonempty supervisor set confirmed by automated extraction exactly matches the complete manual validation supervisor set. | Theses with an exact supervisor-set match / eligible theses with at least one supervisor ID accepted by automated extraction. Incomplete or additional-ID sets count as incorrect. |
| All supervisors found rate | How often automated extraction recovers every manual validation supervisor, allowing additional IDs. | Fraction of eligible theses where automated extraction accepts every supervisor ID in manual validation; additional automated extraction IDs are allowed. |
| End-to-end link precision | How reliable complete supervision relationships in automated extraction are against manual validation. | Matching supervisor pairs where automated extraction also accepts the PhD ID in manual validation / accepted supervisor pairs in automated extraction. |
| End-to-end link recall | How completely automated extraction recovers complete supervision relationships in manual validation. | Matching supervisor pairs where automated extraction also accepts the PhD ID in manual validation / supervisor pairs in manual validation. |
| End-to-end link F1 | Balance between automated extraction link precision and recall against manual validation. | Twice the correct end-to-end link count / (accepted supervisor pairs in automated extraction + supervisor pairs in manual validation). |

Supervisor and link measures require manual validation to supply a PhD ID
and a resolved supervisor list. **Micro averaging** pools pair counts
across eligible theses before dividing for individual supervisors and links;
the all-supervisors measures count theses instead of individual pairs.
A zero denominator makes a measure undefined. The all supervisors found
rate allows additional automated extraction IDs, while all-supervisors
recall requires an exact set match.

Manual validation and automated extraction differ in thesis coverage; the
difference is reported once below. All matching measures here use theses
present in both manual validation and automated extraction and all supervisor
IDs in manual validation, even when automated extraction considers only two
contributors.

The **recall figure is shown first**, followed by a separate **precision
figure**. Both show automated extraction agreement with manual validation,
ordered by PhDs, individual supervisors, all supervisors, then full links.
Higher recall, precision, F1,
and supervisor-set agreement rates are better. A dotted vertical line at
1.0, labeled "manual validation", marks the comparison benchmark in both figures.

The separate confirmation-coverage figure shows **confirmation coverage**
for manual validation and automated extraction. For manual validation, a confirmed identity has a recorded
OpenAlex ID. For automated extraction, a confirmed identity has both a
confirmation decision and a recorded OpenAlex ID, whether or not that ID
agrees with manual validation.

- **PhDs confirmed:** the fraction of theses with a confirmed PhD ID in manual
  validation and automated extraction, respectively.
- **Individual supervisor entries confirmed:** the fraction of listed
  supervisor entries with IDs in manual validation, and the fraction of
  considered contributor rows confirmed with IDs in automated extraction.
  Each automated extraction contributor row counts once, even if it contains
  several OpenAlex IDs.
- **All listed supervisors confirmed:** the fraction of theses with a
  nonempty supervisor list and IDs for every entry in manual validation,
  and the fraction with every considered contributor confirmed with IDs in
  automated extraction. A thesis with no listed supervisors is not counted
  as fully confirmed.
- **End-to-end discovery confirmed:** the fraction of theses where both the
  PhD and every listed supervisor have confirmed IDs in manual validation or
  automated extraction, respectively. Both conditions must hold for the same
  thesis. This is confirmation coverage within each dataset's own list;
  agreement between manual validation and automated extraction is assessed
  separately by recall and precision.

Counts and denominators appear on the bars. These coverage fractions describe
completeness within each dataset's own list: manual validation can list up to
five supervisors, while automated extraction considers contributors selected
by the matching configuration. "All listed" therefore does not mean all
supervisors in manual validation were recovered by automated extraction;
that comparison is the **all supervisors found rate** defined above.

Higher confirmation coverage is better for completeness. These bars show how
complete manual validation is, rather than independently verifying that its
recorded IDs are correct. Recall and precision describe agreement separately.

In [ ]:
manual_validation_ids = set(manual_validation.integer_id)
automated_extraction_ids = set(automated_extraction.integer_id)
print(
    f"Thesis coverage difference: manual validation has {len(manual_validation_ids)} theses; "
    f"automated extraction has {len(automated_extraction_ids)}. "
    f"{len(manual_validation_ids - automated_extraction_ids)} theses in manual validation are absent from automated extraction; "
    f"{len(automated_extraction_ids - manual_validation_ids)} are present only in automated extraction. "
    f"Matching measures use the {len(manual_validation_ids & automated_extraction_ids)} theses present in both."
)

AUTOMATED_COMPARISON = "Automated extraction compared with manual validation"
summary = metric_summary(shared_thesis_comparison).to_frame(AUTOMATED_COMPARISON)

# Score whole supervisor sets on the same eligible theses as individual supervisors.
supervisor_set_comparison = shared_thesis_comparison.loc[
    shared_thesis_comparison.reference_phd_ids.map(bool)
    & shared_thesis_comparison.reference_supervisors_complete
]
n_matching_supervisor_sets = int(supervisor_set_comparison.exact_supervisor_set.sum())
n_confirmed_supervisor_sets = int(supervisor_set_comparison.n_predicted_ids.gt(0).sum())
summary.loc["all_supervisors_recall", AUTOMATED_COMPARISON] = ratio(
    n_matching_supervisor_sets, len(supervisor_set_comparison)
)
summary.loc["all_supervisors_precision", AUTOMATED_COMPARISON] = ratio(
    n_matching_supervisor_sets, n_confirmed_supervisor_sets
)
phd_rates = ["phd_precision", "phd_recall", "phd_f1", "phd_candidate_recall"]
supervisor_rates = ["supervisor_id_precision", "supervisor_id_recall", "supervisor_id_f1",
                    "all_supervisors_recall", "all_supervisors_precision",
                    "exact_supervisor_set_rate", "all_supervisors_found_rate"]
link_rates = ["link_precision", "link_recall", "link_f1"]
rate_names = phd_rates + supervisor_rates + link_rates
count_names = [name for name in summary.index if name.startswith("n_") and name != "n_extracted"]
ordered_counts = [name for name in count_names if name != "n_link_hits"] + ["n_link_hits"]
summary = summary.reindex([name for name in summary.index if name.startswith("n_")] + rate_names)
tables["metric_summary"] = summary
print("Automated extraction counts compared with manual validation: PhDs, supervisors, then full links")
display(dataset_table(summary.loc[ordered_counts]).style.format("{:,.0f}", na_rep="undefined"))
print("Automated extraction agreement with manual validation (higher is better)")
display(dataset_table(summary.loc[rate_names]).style.format("{:.1%}", na_rep="undefined"))

plot_metrics = ["phd_precision", "phd_recall", "supervisor_id_precision", "supervisor_id_recall",
                "all_supervisors_recall", "all_supervisors_precision", "link_precision", "link_recall"]
metric_plot = summary.loc[plot_metrics].copy()
tables["precision_recall_overview"] = metric_plot

# Confirmation coverage uses each dataset's own listed people, not ID agreement.
automated_extraction_shared = automated_extraction.loc[
    automated_extraction.integer_id.isin(shared_thesis_comparison.integer_id)
].copy()
automated_extraction_phds = automated_extraction_shared.drop_duplicates("integer_id")
automated_phd_confirmed = (
    automated_extraction_phds.phd_confirmed
    & automated_extraction_phds.candidate_phd_ids.map(bool)
)
automated_contributors_listed = automated_extraction_shared.loc[
    automated_extraction_shared.contributor_name_narcis.astype(str).str.strip().ne("")
].copy()
automated_contributors_listed["confirmed_with_id"] = (
    automated_contributors_listed.contributor_confirmed
    & automated_contributors_listed.candidate_supervisor_ids.map(bool)
)
automated_all_supervisors_confirmed = automated_contributors_listed.groupby("integer_id")["confirmed_with_id"].all()
n_comparison_theses = len(shared_thesis_comparison)
manual_supervisor_entries = int(shared_thesis_comparison.n_manual_supervisors.sum())
manual_supervisor_entries_with_ids = int((
    shared_thesis_comparison.n_manual_supervisors - shared_thesis_comparison.n_unresolved_supervisors
).sum())

confirmation_counts = pd.DataFrame({
    "Manual validation confirmed": [
        int(shared_thesis_comparison.reference_phd_ids.map(bool).sum()),
        manual_supervisor_entries_with_ids,
        int(shared_thesis_comparison.reference_supervisors_complete.sum()),
    ],
    "Manual validation total": [n_comparison_theses, manual_supervisor_entries, n_comparison_theses],
    "Automated extraction confirmed": [
        int(automated_phd_confirmed.sum()),
        int(automated_contributors_listed.confirmed_with_id.sum()),
        int(automated_all_supervisors_confirmed.reindex(
            shared_thesis_comparison.integer_id, fill_value=False
        ).sum()),
    ],
    "Automated extraction total": [n_comparison_theses, len(automated_contributors_listed), n_comparison_theses],
}, index=["PhDs confirmed", "Individual supervisor entries confirmed", "All listed supervisors confirmed"])
# Align confirmation flags by thesis ID before requiring both parts of discovery.
manual_end_to_end_confirmed = (
    shared_thesis_comparison.reference_phd_ids.map(bool)
    & shared_thesis_comparison.reference_supervisors_complete
)
automated_phd_confirmed_by_thesis = pd.Series(
    automated_phd_confirmed.to_numpy(), index=automated_extraction_phds.integer_id
)
automated_end_to_end_confirmed = (
    automated_phd_confirmed_by_thesis
    & automated_all_supervisors_confirmed.reindex(automated_phd_confirmed_by_thesis.index, fill_value=False)
)
confirmation_counts.loc["End-to-end: PhD and all listed supervisors confirmed"] = {
    "Manual validation confirmed": int(manual_end_to_end_confirmed.sum()),
    "Manual validation total": n_comparison_theses,
    "Automated extraction confirmed": int(automated_end_to_end_confirmed.reindex(
        shared_thesis_comparison.integer_id, fill_value=False
    ).sum()),
    "Automated extraction total": n_comparison_theses,
}

confirmation_coverage = pd.DataFrame({
    "Automated extraction": confirmation_counts["Automated extraction confirmed"].div(
        confirmation_counts["Automated extraction total"].replace(0, np.nan)),
    "Manual validation": confirmation_counts["Manual validation confirmed"].div(
        confirmation_counts["Manual validation total"].replace(0, np.nan)),
})
tables["confirmation_coverage_fractions"] = confirmation_coverage
show_table("confirmation_coverage_counts", confirmation_counts)

fig, ax = plt.subplots(figsize=(11, 7))
confirmation_coverage.plot.barh(ax=ax, color=["tab:blue", "#9a9a9a"])
ax.invert_yaxis()
ax.set(xlim=(0, 1), xlabel="Fraction confirmed (higher is better)",
            title="Confirmation coverage within each dataset's own list")
for bars, source in zip(ax.containers, confirmation_coverage.columns):
    confirmed = confirmation_counts[f"{source} confirmed"]
    total = confirmation_counts[f"{source} total"]
    ax.bar_label(bars, labels=[
        f"{value:.1%} ({int(numerator)}/{int(denominator)})" if pd.notna(value) else "Undefined (0 entries)"
        for value, numerator, denominator in zip(confirmation_coverage[source], confirmed, total)
    ], padding=3, fontsize=8)
show_plot("confirmation_coverage_overview", fig)


for figure_name, metric_names, measure in [
    ("recall_overview", ["phd_recall", "supervisor_id_recall", "all_supervisors_recall", "link_recall"], "Recall"),
    ("precision_overview", ["phd_precision", "supervisor_id_precision", "all_supervisors_precision", "link_precision"], "Precision"),
]:
    measure_plot = metric_plot.loc[metric_names]
    tables[figure_name] = measure_plot
    fig, ax = plt.subplots(figsize=(11, 6))
    measure_plot.plot.barh(ax=ax, color=["tab:blue"], legend=False)
    ax.invert_yaxis()
    ax.set(xlim=(0, 1), xlabel=f"{measure} (higher is better)",
           title="PhDs, supervisors, then full links")
    ax.set_xticks(np.linspace(0, 1, 6))
    ax.axvline(1.0, color="#666666", linestyle=":", linewidth=1.6, zorder=0, clip_on=False)
    ax.annotate("manual validation", xy=(1.0, 1.0), xycoords=ax.get_xaxis_transform(),
                xytext=(-5, -5), textcoords="offset points", ha="right", va="top",
                fontsize=9, color="#666666")
    ax.set_yticks(ax.get_yticks(), [METRIC_LABELS[name] for name in metric_names])
    for bars in ax.containers:
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    show_plot(figure_name, fig)

# Calculate correction sensitivity now; show each level in its corresponding section.
manual_validation_before_corrections, _ = prepare_manual(manual_validation_clean, {})
comparison_before_manual_validation_corrections, _ = compare_identities(manual_validation_before_corrections, automated_extraction)
manual_validation_correction_sensitivity = pd.DataFrame({
    "Manual validation before supervisor ID corrections": metric_summary(
        comparison_before_manual_validation_corrections.loc[comparison_before_manual_validation_corrections.in_automated]),
    "Manual validation after supervisor ID corrections from notes": metric_summary(shared_thesis_comparison),
})
tables["reference_correction_sensitivity"] = manual_validation_correction_sensitivity.loc[
    ["n_identity_hits", "supervisor_id_precision", "supervisor_id_recall", "n_link_hits", "link_precision", "link_recall"]
]

## 6. PhD-level agreement between automated extraction and manual validation

These results count PhD records, one per thesis in both manual validation
and automated extraction. An **evidence matrix** counts combinations of
automated extraction PhD evidence categories and PhD-ID agreement with manual
validation. Fractions divide each count by the PhD-record count in that
evidence category. Higher correct-ID agreement is better; lower disagreement
and unconfirmed counts are better. Missing manual validation IDs are unscored.

In [ ]:
OUTCOME_COLORS = {PHD_OUTCOMES[0]: "#2a9d8f", PHD_OUTCOMES[1]: "#e76f51",
                  PHD_OUTCOMES[2]: "#e9c46a", PHD_OUTCOMES[3]: "#79706e",
                  PHD_OUTCOMES[4]: "#b8b8d1"}
phd_display_outcomes = [outcome for outcome in PHD_OUTCOMES if outcome != "Missing extraction"]
outcome_counts = shared_thesis_comparison.phd_outcome.value_counts().reindex(phd_display_outcomes, fill_value=0)
show_table("phd_outcomes", outcome_counts.rename("n_theses").to_frame())
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(outcome_counts.index, outcome_counts.values, color=[OUTCOME_COLORS[v] for v in outcome_counts.index])
ax.bar_label(bars, padding=3)
ax.invert_yaxis()
ax.set(xlabel="Number of PhD records (one per thesis)", title="PhD-level ID agreement with manual validation")
fig.tight_layout()
show_plot("phd_identity_outcomes", fig)

phd_evidence_counts = pd.crosstab(shared_thesis_comparison.phd_evidence_category, shared_thesis_comparison.phd_outcome).reindex(
    index=PhDMatchPlotter._BAR_CATEGORIES, columns=phd_display_outcomes, fill_value=0,
)
phd_evidence_fractions = phd_evidence_counts.div(phd_evidence_counts.sum(axis=1).replace(0, np.nan), axis=0)
show_table("phd_outcomes_by_evidence_counts", phd_evidence_counts)
show_table("phd_outcomes_by_evidence_fractions", phd_evidence_fractions)
fig, ax = plt.subplots(figsize=(13, 7))
phd_evidence_counts.plot.barh(stacked=True, ax=ax, color=[OUTCOME_COLORS[c] for c in phd_evidence_counts])
ax.set(xlabel="Number of PhD records (one per thesis)", ylabel="PhD matching evidence in automated extraction",
       title="PhD-level ID agreement by automated extraction evidence")
ax.invert_yaxis()
ax.legend(loc="upper left", bbox_to_anchor=(1, 1), fontsize=8)
fig.tight_layout()
show_plot("phd_agreement_by_evidence", fig)

### Automated extraction match scores and title similarity

Compare automated extraction scores for PhD IDs that agree with manual
validation, PhD IDs that differ from manual validation, and PhDs not
confirmed by automated extraction. The **retention curve** shows precision
and recall against manual validation after keeping only confirmed PhDs
from automated extraction above each score cutoff. It cannot add
identities that automated extraction did not confirm. In the table,
`-inf` applies no cutoff and `inf` keeps no finite-score matches.

This **sensitivity analysis** describes how filtering automated extraction
changes agreement with the current manual validation; it does not
evaluate the cutoff on a separate manual validation dataset.

In [ ]:
score_diagnostics = shared_thesis_comparison.loc[shared_thesis_comparison.reference_phd_ids.map(bool)].copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, column, label in zip(
    axes, ["auto_phd_match_score", "auto_max_title_similarity"],
    ["PhD match score in automated extraction", "Maximum title similarity in automated extraction"],
):
    all_values = pd.to_numeric(score_diagnostics[column], errors="coerce").dropna()
    bins = np.histogram_bin_edges(all_values, bins=20) if not all_values.empty else 20
    for outcome in PHD_OUTCOMES[:3]:
        values = pd.to_numeric(score_diagnostics.loc[score_diagnostics.phd_outcome.eq(outcome), column], errors="coerce").dropna()
        if not values.empty:
            ax.hist(values, bins=bins, alpha=0.55, label=f"{outcome} (n={len(values)})", color=OUTCOME_COLORS[outcome])
    ax.set(xlabel=label, ylabel="Frequency of PhD records (descriptive)")
    ax.legend(fontsize=7)
fig.tight_layout()
show_plot("phd_score_distributions", fig)

accepted = score_diagnostics.loc[score_diagnostics.phd_confirmed].copy()
accepted["score"] = pd.to_numeric(accepted.auto_phd_match_score, errors="coerce")
thresholds = [-np.inf, *sorted(accepted.score.dropna().unique()), np.inf]
retention_rows = []
for threshold in thresholds:
    retained = accepted if threshold == -np.inf else accepted.loc[accepted.score.ge(threshold)]
    correct = int(retained.phd_correct.sum())
    retention_rows.append({
        "minimum_score": threshold, "n_retained": len(retained), "n_correct": correct,
        "n_wrong": len(retained) - correct,
        "precision": ratio(correct, len(retained)),
        "recall": ratio(correct, len(score_diagnostics)),
    })
retention = pd.DataFrame(retention_rows)
show_table("phd_score_retention", retention)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(retention.recall, retention.precision, marker=".", color="tab:blue")
ax.set(xlim=(0, 1), ylim=(0, 1.03), xlabel="Recall against manual validation (higher is better)", ylabel="Precision against manual validation (higher is better)",
       title="Confirmed automated extraction PhDs retained above a score cutoff")
ax.grid(alpha=0.2)
fig.tight_layout()
show_plot("phd_score_retention_curve", fig)

### ORCID agreement between automated extraction and manual validation

**ORCID concordance** checks whether manual validation and automated
extraction share an ORCID for a PhD confirmed by automated extraction.
Compare only theses with usable ORCIDs in both manual validation and
automated extraction. An agreeing ORCID with differing OpenAlex IDs may
indicate a profile change or an identity-selection issue; it does not
change the primary OpenAlex-ID comparison. Missing ORCIDs are excluded.

In [ ]:
orcid_comparable = shared_thesis_comparison.loc[shared_thesis_comparison.phd_confirmed & shared_thesis_comparison.phd_orcid_comparable]
orcid_counts = pd.crosstab(orcid_comparable.phd_outcome, orcid_comparable.phd_orcid_agrees).reindex(columns=[False, True], fill_value=0)
orcid_counts.columns = ["ORCID differs between manual validation and automated extraction", "ORCID agrees between manual validation and automated extraction"]
show_table("phd_orcid_concordance", orcid_counts)
possible_profile_changes = orcid_comparable.loc[
    orcid_comparable.phd_outcome.eq("Wrong confirmed ID") & orcid_comparable.phd_orcid_agrees,
    ["integer_id", "author_name", "reference_phd_ids", "predicted_phd_ids", "reference_phd_orcids", "notes"],
]
show_table("different_openalex_id_same_orcid", possible_profile_changes)

### Review PhD-level disagreements

Inspect automated extraction PhD IDs that disagree with manual validation, including confirmations and unconfirmed candidates.

In [ ]:
phd_review_columns = ["integer_id", "author_name", "auto_phd_name", "thesis_identifier", "institution", "year",
                      "reference_phd_ids", "candidate_phd_ids", "phd_outcome", "auto_phd_match_score",
                      "auto_max_title_similarity", "phd_evidence_category", "processing_error", "notes"]
phd_review = shared_thesis_comparison.loc[
    shared_thesis_comparison.reference_phd_ids.map(bool) & ~shared_thesis_comparison.phd_correct, phd_review_columns,
].copy()
phd_review["sort_score"] = pd.to_numeric(phd_review.auto_phd_match_score, errors="coerce")
phd_review = phd_review.sort_values(["phd_outcome", "sort_score"], ascending=[True, False]).drop(columns="sort_score")
show_table("phd_disagreements", phd_review)

## 7. Supervisor-level agreement between automated extraction and manual validation

Compare supervisor IDs first, independently of PhD-ID agreement. Complete
PhD-supervisor links are evaluated next in section 8. The first plot describes
automated extraction evidence; the following plots check those IDs against
manual validation. Evidence frequencies alone have no better/worse direction.

In [ ]:
ax = contributor_plotter.plot()
ax.set_title(
    f"{contributor_plotter.n_phds_with_confirmed} PhD records have a contributor confirmed by automated extraction\n"
    f"{contributor_plotter.n_confirmed_contributors} of {contributor_plotter.n_contributors} contributors are confirmed"
)
show_plot("contributor_extraction_evidence", ax.figure)

### Automated extraction contributor evidence, rank, and multiple identities

These tables count contributor rows from automated extraction rather than
distinct supervisor IDs. **Matches manual validation + additional IDs**
means automated extraction includes at least one supervisor ID from manual
validation and at least one other ID. **All IDs match manual validation**
means every accepted automated extraction ID occurs in manual validation.
**Manual validation incomplete (unscored)** means an accepted automated
extraction row has no known matching ID, but missing supervisor IDs in
manual validation prevent scoring it as incorrect.

**Contributor match rate against manual validation** is the fraction of
confirmed automated extraction contributor rows with resolved manual
validation supervisor lists that include at least one manual validation
supervisor ID. A row with additional automated extraction IDs still counts
as a row-level match; supervisor ID precision counts those additional IDs
separately.

**Multi-ID rows** contain several distinct IDs for one contributor in
automated extraction. **Contributor scope** is the number of contributors
considered by automated extraction per thesis; comparing it with the
supervisor count in manual validation indicates selection limits rather
than a strict numerical ceiling on recall.

In [ ]:
contributor_outcomes = ["Reference IDs only", "Reference hit + extra IDs", "No reference hit",
                        "Not confirmed", "Confirmed without ID", "Incomplete reference (unscored)"]
contributor_colors = ["#2a9d8f", "#e9c46a", "#e76f51", "#aaaaaa", "#9b5de5", "#b8b8d1"]
contributor_evidence = pd.crosstab(contributor_comparison.evidence_category, contributor_comparison.identity_outcome).reindex(
    index=ContributorMatchPlotter._BAR_CATEGORIES, columns=contributor_outcomes, fill_value=0,
)
show_table("contributor_identity_by_evidence", contributor_evidence)
fig, ax = plt.subplots(figsize=(12, 5))
contributor_evidence.plot.barh(stacked=True, ax=ax, color=contributor_colors)
ax.set(xlabel="Contributor rows", ylabel="", title="Automated extraction contributor evidence compared with manual validation")
ax.invert_yaxis()
ax.legend(loc="upper left", bbox_to_anchor=(1, 1), fontsize=8)
fig.tight_layout()
show_plot("contributor_reference_agreement_by_evidence", fig)

rank_rows = []
for rank, group in contributor_comparison.groupby("contributor_rank", dropna=False, sort=True):
    scored = group.loc[group.reference_supervisors_complete & group.contributor_confirmed]
    rank_rows.append({
        "contributor_rank": rank, "n_contributor_rows": len(group),
        "n_confirmed": int(group.contributor_confirmed.sum()),
        "n_confirmed_with_complete_reference": len(scored),
        "n_reference_hit_rows": int(scored.matched_reference_ids.map(bool).sum()),
        "reference_hit_rate_among_scored_confirmations": ratio(int(scored.matched_reference_ids.map(bool).sum()), len(scored)),
        "n_confirmed_multi_ID_rows": int((group.contributor_confirmed & group.n_candidate_ids.gt(1)).sum()),
    })
show_table("contributor_rank_diagnostics", pd.DataFrame(rank_rows), origin="automated extraction")

scope_counts = shared_thesis_comparison.groupby(["n_manual_supervisors", "n_contributor_rows"]).agg(
    n_theses=("integer_id", "size"), n_reference_ids=("n_reference_ids", "sum"),
    n_identity_hits=("n_identity_hits", "sum"),
).reset_index()
show_table("manual_supervisor_count_vs_contributor_scope", scope_counts)
print(f"{int((shared_thesis_comparison.n_manual_supervisors > shared_thesis_comparison.n_contributor_rows).sum())} theses present in both manual validation and automated extraction have more supervisors in manual validation than contributor rows considered by automated extraction.")
print("This indicates automated extraction contributor scope; several IDs per automated extraction row mean it is not a strict ceiling on recall against manual validation.")

### Supervisor IDs and complete supervisor sets

Count each thesis-supervisor ID pair once. Higher matching-ID counts are
better; lower additional-ID and missed-ID counts are better. For complete
supervisor sets, more exact matches to manual validation are better. Partial
sets are incomplete outcomes rather than an alternative definition of recall.

**Manual validation correction sensitivity** compares supervisor measures
before and after applying the two supervisor-ID corrections from manual
validation notes.

In [ ]:
relation_cohort = shared_thesis_comparison.loc[
    shared_thesis_comparison.reference_phd_ids.map(bool) & shared_thesis_comparison.reference_supervisors_complete
].copy()
supervisor_counts = pd.DataFrame({
    "Automated extraction supervisor IDs": [
        relation_cohort.n_identity_hits.sum(),
        (relation_cohort.n_predicted_ids - relation_cohort.n_identity_hits).sum(),
        (relation_cohort.n_reference_ids - relation_cohort.n_identity_hits).sum(),
    ],
}, index=["Reference hits", "Extra / incorrect predictions", "Missed reference relations"])
tables["supervisor_identity_counts"] = supervisor_counts

def set_outcome(row):
    if row.exact_supervisor_set:
        return "Exact supervisor set"
    if row.all_reference_supervisors_found:
        return "All reference IDs + extra IDs"
    if row.n_identity_hits:
        return "Some reference supervisors found"
    return "No reference hits" if row.n_predicted_ids else "No accepted supervisor IDs"

set_order = ["Exact supervisor set", "All reference IDs + extra IDs", "Some reference supervisors found",
             "No reference hits", "No accepted supervisor IDs"]
relation_cohort["supervisor_set_outcome"] = relation_cohort.apply(set_outcome, axis=1)
set_counts = relation_cohort.supervisor_set_outcome.value_counts().reindex(set_order, fill_value=0)
show_table("supervisor_set_outcomes", set_counts.rename("n_theses").to_frame())
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
bars = axes[0].barh(supervisor_counts.index, supervisor_counts.iloc[:, 0], color=["#2a9d8f", "#e76f51", "#e9c46a"])
axes[0].bar_label(bars, padding=3)
axes[0].invert_yaxis()
axes[0].set(xlabel="Thesis-supervisor ID pairs", title="Supervisor-ID agreement")
bars = axes[1].barh(set_counts.index, set_counts.values, color=["#2a9d8f", "#e9c46a", "#e9c46a", "#e76f51", "#aaaaaa"])
axes[1].bar_label(bars, padding=3)
axes[1].invert_yaxis()
axes[1].set(xlabel="Number of theses", title="Agreement of complete supervisor sets")
show_plot("supervisor_identity_and_set_agreement", fig)
show_table("reference_correction_sensitivity", manual_validation_correction_sensitivity.loc[
    ["n_identity_hits", "supervisor_id_precision", "supervisor_id_recall"]
])

### Review supervisor-level disagreements

Review automated extraction supervisor IDs against manual validation before considering complete PhD-supervisor links.

In [ ]:
supervisor_review_columns = ["integer_id", "author_name", "thesis_identifier", "phd_outcome", "reference_supervisor_ids",
    "predicted_supervisor_ids", "missed_supervisor_ids", "extra_supervisor_ids",
    "n_manual_supervisors", "n_contributor_rows", "reference_supervisors_complete", "processing_error", "notes"]
show_table("missed_manual_supervisors", shared_thesis_comparison.loc[shared_thesis_comparison.missed_supervisor_ids.map(bool), supervisor_review_columns])
show_table("extra_supervisor_predictions", shared_thesis_comparison.loc[
    shared_thesis_comparison.reference_supervisors_complete & shared_thesis_comparison.extra_supervisor_ids.map(bool), supervisor_review_columns,
])
ambiguous_columns = ["integer_id", "phd_name", "contributor_rank", "contributor_name_narcis", "contributor_name",
                     "candidate_supervisor_ids", "matched_reference_ids", "extra_candidate_ids", "identity_outcome",
                     "phd_correct", "evidence_category", "sup_match_by", "n_shared_pubs", "n_shared_inst_grad"]
show_table("ambiguous_contributor_identities", contributor_comparison.loc[
    contributor_comparison.contributor_confirmed & contributor_comparison.n_candidate_ids.gt(1), ambiguous_columns,
].sort_values(["identity_outcome", "integer_id"]), origin="automated extraction")
show_table("confirmed_contributors_without_reference_hit", contributor_comparison.loc[
    contributor_comparison.contributor_confirmed & ~contributor_comparison.matched_reference_ids.map(bool), ambiguous_columns,
], origin="automated extraction")

tables["thesis_level_diagnostics"] = thesis_comparison
tables["contributor_level_diagnostics"] = contributor_comparison

## 8. Full-link agreement between automated extraction and manual validation

An **end-to-end link** is correct only when automated extraction confirms
both the PhD ID and its supervisor ID and both match manual validation for
the same thesis. This follows the PhD-level and supervisor-level results.

Higher complete-link agreement is better. Lower incorrect-link and
missed-link counts are better. A matching supervisor ID with a different
or unconfirmed PhD ID is not a correct complete link.

In [ ]:
relation_counts = supervisor_counts.rename(columns={
    "Automated extraction supervisor IDs": "Supervisor IDs (regardless of PhD agreement)"
}).copy()
relation_counts["End-to-end PhD-supervisor links"] = [
    relation_cohort.n_link_hits.sum(),
    (relation_cohort.n_predicted_ids - relation_cohort.n_link_hits).sum(),
    (relation_cohort.n_reference_ids - relation_cohort.n_link_hits).sum(),
]
show_table("supervision_relation_counts", relation_counts)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(relation_counts.index, relation_counts.iloc[:, 1], color=["#2a9d8f", "#e76f51", "#e9c46a"])
ax.bar_label(bars, padding=3)
ax.invert_yaxis()
ax.set(xlabel="Complete PhD-supervisor links", title="Full-link agreement requires both identities to match")
show_plot("end_to_end_link_agreement", fig)
show_table("link_correction_sensitivity", manual_validation_correction_sensitivity.loc[
    ["n_link_hits", "link_precision", "link_recall"]
])
link_review_columns = ["integer_id", "author_name", "phd_outcome", "reference_phd_ids", "predicted_phd_ids",
                       "reference_supervisor_ids", "predicted_supervisor_ids", "missed_link_ids", "notes"]
show_table("missed_manual_validation_links", shared_thesis_comparison.loc[
    shared_thesis_comparison.missed_link_ids.map(bool), link_review_columns
])

## 9. Automated extraction agreement by manual validation institution, year, and completeness

Group automated extraction agreement using institution, thesis year,
accessibility, and notes from manual validation. Each group retains its
thesis count and the count eligible for supervisor/link comparison.
Accessibility and notes in manual validation define groups rather than
automatic exclusions.

**Conditional on PhD ID agreement** repeats the supervisor comparison
only for theses where the PhD ID accepted by automated extraction matches
manual validation. This separates supervisor matching after a successful
PhD match from performance across all theses present in both manual
validation and automated extraction.

In [ ]:
group_columns = ["n_theses", "n_phd_labeled", "phd_precision", "phd_recall",
                 "n_relation_theses", "supervisor_id_precision", "supervisor_id_recall", "link_recall"]
institution_metrics = group_metrics(shared_thesis_comparison, "institution")
year_metrics = group_metrics(shared_thesis_comparison, "year").sort_index()
show_table("metrics_by_institution", institution_metrics[group_columns], rows=30)
show_table("metrics_by_year", year_metrics[group_columns], rows=30)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
institution_metrics[["phd_recall", "supervisor_id_recall", "link_recall"]].plot.barh(ax=axes[0])
axes[0].set(xlim=(0, 1), xlabel="Recall (higher is better)", ylabel="Institution in manual validation", title="Automated extraction recall by manual validation institution")
axes[0].legend(fontsize=7)
year_metrics[["phd_recall", "supervisor_id_recall", "link_recall"]].plot(marker="o", ax=axes[1])
axes[1].set(ylim=(0, 1), ylabel="Recall (higher is better)", xlabel="Thesis year in manual validation", title="Automated extraction recall by manual validation thesis year")
axes[1].legend(fontsize=7)
fig.tight_layout()
show_plot("recall_by_institution_and_year", fig)

quality_strata = shared_thesis_comparison.copy()
quality_strata["accessibility"] = quality_strata.thesis_correct_and_accessible.str.strip().replace("", "unrecorded")
quality_strata["manual_notes"] = np.where(quality_strata.notes.str.strip().ne(""), "Notes present in manual validation", "No notes in manual validation")
show_table("metrics_by_thesis_accessibility", group_metrics(quality_strata, "accessibility")[group_columns])
show_table("metrics_by_manual_notes", group_metrics(quality_strata, "manual_notes")[group_columns])

# Compare automated extraction supervisor matching after PhD ID agreement with manual validation.
comparison_with_matching_phd_ids = shared_thesis_comparison.loc[shared_thesis_comparison.phd_correct]
show_table("conditional_on_correct_phd", pd.DataFrame({
    "Theses in both manual validation and automated extraction": metric_summary(shared_thesis_comparison),
    "Theses in manual validation and automated extraction with agreeing confirmed PhD ID": metric_summary(comparison_with_matching_phd_ids),
}).loc[["n_theses", "n_relation_theses", "supervisor_id_precision", "supervisor_id_recall", "link_recall"]])

## 10. Export the comparison of automated extraction and manual validation

This notebook exports complete comparison tables as CSVs and figures as
PNGs. The manifest records the manual validation and automated extraction
file hashes, corrections applied to manual validation, package versions,
and the theses included in each measure. Reports with the same names are
replaced when this cell runs. Manual validation, automated extraction,
and the matching configuration are not edited.

In [ ]:
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "inputs": provenance.to_dict("records"),
    "config_path": str(CONFIG_PATH),
    "configured_contributor_rank_limit": config.get("min_rank_contrib"),
    "observed_contributor_ranks": sorted(automated_extraction.contributor_rank.unique().tolist()),
    "alternative_supervisor_slots": ALTERNATIVE_SUPERVISOR_SLOTS,
    "recovered_manual_keys": key_recovery.to_dict("records"),
    "identifier_quality_audit_rows": len(identifier_audit),
    "primary_cohort": "Theses present in both manual validation and automated extraction after audited manual validation ID recovery",
    "confirmation_coverage": "PhDs per thesis; individual supervisor entries per dataset own list; all listed supervisors per thesis; end-to-end discovery requires both the PhD and all listed supervisors confirmed for the same thesis; confirmations require IDs and automated extraction also requires confirmation decisions",
    "confirmation_coverage_scope": "Manual validation supervisor entries and automated extraction considered contributors can differ; coverage does not measure ID correctness or recovery of the complete manual validation supervisor set",
    "display_order": ["PhD identities", "Supervisor identities and sets", "End-to-end PhD-supervisor links"],
    "supervisor_scope": "Resolved nonempty supervisor lists and known PhD IDs in manual validation; all five manual validation supervisor slots",
    "supervisor_id_metric": "Pooled individual supervisor ID agreement between automated extraction and manual validation, regardless of PhD ID agreement",
    "all_supervisors_metric": "Exact equality of the nonempty automated extraction supervisor ID set with the complete manual validation set, regardless of PhD ID agreement; recall divides exact matches by eligible theses, precision divides exact matches by eligible theses with at least one confirmed automated extraction supervisor ID",
    "link_metric": "Pooled supervisor ID agreement between automated extraction and manual validation, also requiring a confirmed automated extraction PhD ID that matches manual validation",
    "multiple_returned_ids": "Every accepted automated extraction ID is evaluated; IDs absent from resolved manual validation lists reduce precision",
    "missing_labels": "IDs missing from manual validation are unknown and excluded from the applicable metric denominator",
    "notes": "Exact ID agreement between automated extraction and manual validation; an ID absent from manual validation can still require review",
    "versions": {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__},
}
if WRITE_REPORTS:
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    for name, frame in tables.items():
        export_frame(frame).to_csv(REPORT_DIR / f"{name}.csv", index=True, encoding="utf-8-sig")
    for name, figure in figures.items():
        figure.savefig(REPORT_DIR / f"{name}.png", dpi=160, bbox_inches="tight")
    (REPORT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Wrote {len(tables)} complete CSV tables, {len(figures)} figures, and manifest.json to {REPORT_DIR}")
else:
    print("Reports were retained in the tables and figures dictionaries; no files were written.")